In [ ]:
from IPython.display import Markdown
import torch
import torch.nn.functional as F
import string
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

%matplotlib inline

## data

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    with open("/content/drive/MyDrive/usnames.txt", "r") as f:
        names = f.readlines()
except:
    with open("../data/usnames.txt", "r") as f:
        names = f.readlines()
names = [n.strip() for n in names]

In [ ]:
names[:3]

In [ ]:
tok2idx = {c: i for i, c in enumerate('.' + string.ascii_lowercase)}
idx2tok = {i: c for c, i in tok2idx.items()}

In [ ]:
def build_dataset(names, ctx_len=ctx_len):
    X, Y = [], []

    for name in names:
        ctx = ['.'] * ctx_len
        for c in name + '.':
            x = [tok2idx[c] for c in ctx]
            y = tok2idx[c]
            ctx.append(c); ctx.pop(0)
            X.append(x); Y.append(y)

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.shuffle(names)
n1 = int(0.8*len(names))
n2 = int(0.9*len(names))

Xtr, Ytr = build_dataset(names[:n1])
Xdev, Ydev = build_dataset(names[n1:n2])
Xte, Yte = build_dataset(names[n2:])

## modeling

In [ ]:
emb_sz = 10
ctx_len = 3
nhidden = 200

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with `nhidden` neurons
W1 = torch.randn(emb_sz*ctx_len, nhidden, generator=g) * ((5/3)*(emb_sz*ctx_len)**0.5)
b1 = torch.randn(nhidden, generator=g) * 0.01

# output layer
W2 = torch.randn(nhidden, 27, generator=g) * 0.01
b2 = torch.randn(27, generator=g) * 0.01

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

for p in params: p.requires_grad = True

In [ ]:
Xtr, Ytr = build_dataset(names[:n1], ctx_len=ctx_len)
Xdev, Ydev = build_dataset(names[n1:n2], ctx_len=ctx_len)
Xte, Yte = build_dataset(names[n2:], ctx_len=ctx_len)

In [ ]:
steps = []
losses = []
lr = 0.1
for i in tqdm(range(200_000), total=200_000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[Xtr[ixs]]
    preact = X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1
    h = torch.tanh(preact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ixs])

    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    if i > 100_000: lr = 0.01
    if i > 150_000: lr = 0.001
    for p in params:
        p.data += -lr*p.grad

    # track
    steps.append(i)
    losses.append(loss.log10().item())


print(loss.item())

In [ ]:
plt.plot(steps, losses);

In [ ]:
# calculating the loss on full training set

X_emb = word_embeddings[Xtr]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# calculating the loss on validation set

X_emb = word_embeddings[Xdev]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

## sampling from the model

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(25):
    out = []
    ctx = ['.']*ctx_len
    while True:
        emb = word_embeddings[torch.tensor([tok2idx[t] for t in ctx])]
        h = torch.tanh(emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        tok = idx2tok[ix]
        ctx.append(tok); ctx.pop(0)
        out.append(tok)
        if tok == '.': break
    print(''.join(out))

## implementing batch normalisation

In [ ]:
emb_sz = 10
ctx_len = 3
nhidden = 300

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with `nhidden` neurons 
W1 = torch.randn(emb_sz*ctx_len, nhidden, generator=g) * ((5/3)*(emb_sz*ctx_len)**0.5) # kaiming norm
b1 = torch.randn(nhidden, generator=g) * 0.01

# output layer
W2 = torch.randn(nhidden, 27, generator=g) * 0.01
b2 = torch.randn(27, generator=g) * 0.01

# batch norm vars
bngain = torch.ones((1, nhidden))
bnbias = torch.zeros((1, nhidden))
bnmeanrunning = torch.ones((1, nhidden))
bnstdrunning = torch.zeros((1, nhidden))

params = [word_embeddings, W1, b1, W2, b2, bnbias, bngain]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

for p in params: p.requires_grad = True

In [ ]:
Xtr, Ytr = build_dataset(names[:n1], ctx_len=ctx_len)
Xdev, Ydev = build_dataset(names[n1:n2], ctx_len=ctx_len)
Xte, Yte = build_dataset(names[n2:], ctx_len=ctx_len)

In [ ]:
steps = []
losses = []
lr = 0.1
for i in tqdm(range(200_000), total=200_000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[Xtr[ixs]]
    preact = X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1

    # batch norm
    bnmeani = preact.mean(0, keepdim=True)
    bnstdi  = preact.std(0, keepdim=True)
    preact = (preact - bnmeani) / bnstdi
    preact = bngain * preact + bnbias

    # calculating bnmean, bnstd during the training to avoid a calibration step at the end
    # here the 0.001 is the momentum
    with torch.no_grad():
        bnmeanrunning = 0.999*bnmeanrunning + 0.001*bnmeani
        bnstdrunning = 0.999*bnstdrunning + 0.001*bnstdi
    
    h = torch.tanh(preact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ixs])

    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    if i > 100_000: lr = 0.01
    # if i > 150_000: lr = 0.001
    for p in params:
        p.data += -lr*p.grad

    # track
    steps.append(i)
    losses.append(loss.log10().item())

    # break

print(loss.item())

In [ ]:
plt.plot(steps, losses);

In [ ]:
# calculating the loss on full training set

X_emb = word_embeddings[Xtr]
preact = X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1
preact = (preact - bnmeanrunning) / bnstdrunning
preact = bngain * preact + bnbias
h = torch.tanh(preact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# calculating the loss on validation set

X_emb = word_embeddings[Xdev]
preact = X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1
preact = (preact - bnmeanrunning) / bnstdrunning
preact = bngain * preact + bnbias
h = torch.tanh(preact)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

### sampling from the model

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(25):
    out = []
    ctx = ['.']*ctx_len
    while True:
        emb = word_embeddings[torch.tensor([tok2idx[t] for t in ctx])]
        preact = emb.view(-1, emb_sz*ctx_len) @ W1 + b1

        # using claibrated batch norm statistics
        preact = (preact - bnmeanrunning) / bnstdrunning
        preact = bngain * preact + bnbias
        
        h = torch.tanh(preact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        tok = idx2tok[ix]
        ctx.append(tok); ctx.pop(0)
        out.append(tok)
        if tok == '.': break
    print(''.join(out))

## lets torchify the implementation

In [ ]:
emb_sz = 10
ctx_len = 3
nhidden = 300
vocab_sz = len(idx2tok)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_sz, emb_sz), dtype=torch.float32, generator=g)
layers = [torch.nn.Linear(emb_sz*ctx_len, nhidden, bias=False), torch.nn.Tanh(),
          torch.nn.Linear(nhidden, vocab_sz)]

In [ ]:
steps = []
losses = []
total_steps = 200_000
batch_sz = 32
lr = 0.1
for i in tqdm(range(total_steps), total=total_steps):
    ixs = torch.randint(0, Xtr.shape[0], (batch_sz, ))
    embs = C[Xtr[ixs]]
    
    break

In [ ]:
embs.shape